# Bronze Layer: Discover Notebook

**Target Tables:**
- **Read:** Search metadata (`search_locations.yml`)
- **Write:** `bronze_ad_links` (`bronze.GeneralSearch`)

**Objective:**
Discovers raw hardware product URLs from target marketplaces (e.g., OLX). It queries search terms, gathers product listing links, and persists raw links to the Bronze database.

## 1. Setup and Imports
Configure system path, database engine, crawler services, and model namespaces.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.services import OLXCrawler
from app.config import db_engine
from app.models import bronze
from app.utils import read_search_locations_metadata

## 2. Initialize Data Container
Create container list to collect discovered listing link dictionaries.

In [2]:
# Base list for general search results
general_data_rows = []

## 3. Load Location and Store Metadata
Read store target configurations from metadata files.

In [3]:
# Search target location data
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)

stores = (
    brazil_data_location
    .get('stores', [])
)

In [4]:
# Filter configured stores for execution batch
stores = [stores[0]]
print(stores)

## 4. Run Marketplace Crawlers
Execute web crawler loops to discover general listing links.

In [5]:
for store in stores:
    store_name = store.get('name', '')
    store_base_url = store.get('base_url', '')
    
    print(f'Processing discovery for: {store_name}')

    if store_name == 'OLX':
        olx_crawler = OLXCrawler(
            base_url=store_base_url,
            base_data_list=general_data_rows
        )

        await olx_crawler.scrap_general_links()

## 5. Persist Discovered Links to Bronze Database (`bronze_ad_links`)
Construct Polars DataFrame and insert discovered link records into `bronze.GeneralSearch`.

In [6]:
# Construct Polars DataFrame
general_df = pl.DataFrame(general_data_rows)

if not general_df.is_empty():
    with Session(db_engine) as session:
        # Save raw discovered links to SQLite database
        session.execute(insert(bronze.GeneralSearch), general_df.to_dicts())
        session.commit()
        print(f"Successfully persisted {len(general_df)} discovered links to bronze_ad_links.")
else:
    print("No discovery data found to insert.")